# Mission 10 · Stage 5
## Colab 공식 test 1회 전용 노트북

Stage 5 Nested OOF 결과로 다음 설정을 고정했습니다.

```text
입력: header / footer / quote가 제거된 본문
전처리: encoded payload 및 반복 short token 제거
특징: word TF-IDF + character TF-IDF
분류기: Global LinearSVC
Specialist: 사용하지 않음
```

OOF 결과:

| 모델 | Accuracy | Macro F1 |
|---|---:|---:|
| **Nested Global LinearSVC** | **0.7598** | **0.7518** |
| Nested Specialist | 0.7595 | 0.7516 |
| Stage 4 Fixed SGD | 0.7544 | 0.7449 |

이 노트북은 모델 탐색 없이 CV pool 16,019개 전체로 최종 모델을 학습한 뒤, official test 2,827개를 단 한 번 평가합니다.

### 중복 실행 방지

Stage 5 Drive 결과 폴더에서 다음 파일을 확인합니다.

```text
stage5_official_test_running.lock
stage5_official_test_done.lock
final_test_result.json
```

`final_test_result.json` 또는 완료 lock이 있으면 기존 결과만 불러오고 test를 다시 계산하지 않습니다.


## 실행 환경

Stage 5 최종 모델은 sparse LinearSVC이므로 GPU가 필요 없습니다.

```text
Colab 런타임: CPU
하드웨어 가속기: 없음
```


In [1]:
!pip install -q -U scikit-learn google-api-python-client


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 41.2 MB/s eta 0:00:00


In [2]:
import hashlib
import io
import json
import os
import random
import re
import sys
import warnings

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
TEST_SIZE = 0.15

WORD_MAX_FEATURES = 80_000
CHAR_MAX_FEATURES = 120_000

# Stage 5 Drive 폴더
STAGE5_OUTPUT_FOLDER_ID = "1i8veU1sgJ4RVLUQ9PIVYIUX-d42F63TE"
RESULTS_FOLDER_ID = "143oBOq3BjR9lyusBzBJb--RxgOBqmhnB"

RUNNING_LOCK_NAME = "stage5_official_test_running.lock"
DONE_LOCK_NAME = "stage5_official_test_done.lock"
FINAL_RESULT_NAME = "final_test_result.json"
FINAL_REPORT_NAME = "final_test_class_report.csv"
FINAL_CONFUSIONS_NAME = "final_test_top_confusions.csv"
FINAL_PROBABILITIES_NAME = "final_test_probabilities.npy"

# 세션 강제 종료로 running lock만 남았을 때만 True로 바꿉니다.
FORCE_REMOVE_RUNNING_LOCK = False
SAVE_MODEL_TO_DRIVE = False

LOCKED_CONFIG = {
    "stage": 5,
    "input": "body_only_header_footer_quotes_removed",
    "preprocessing": (
        "encoded_payload_and_repeated_short_noise_removal"
    ),
    "classifier": "linear_svc",
    "use_specialist": False,
    "C": 1.0,
    "seed": SEED,
    "test_size": TEST_SIZE,
    "word_ngram_range": [1, 2],
    "char_ngram_range": [3, 5],
    "word_max_features": WORD_MAX_FEATURES,
    "char_max_features": CHAR_MAX_FEATURES,
}

CONFIG_HASH = hashlib.sha256(
    json.dumps(
        LOCKED_CONFIG,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Config hash:", CONFIG_HASH)


Python: 3.12.13
Config hash: c33995409eb8b135ce07a25c841c75513c3968742bc0a3e4c5b435dcfb72446f


## Google Drive 인증과 파일 잠금


In [3]:
from google.colab import auth
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import (
    MediaIoBaseDownload,
    MediaIoBaseUpload,
)

auth.authenticate_user()

credentials, _ = default()

drive_service = build(
    "drive",
    "v3",
    credentials=credentials,
)


def escape_drive_query(value):
    return (
        str(value)
        .replace("\\", "\\\\")
        .replace("'", "\\'")
    )


def find_drive_file(
    name,
    parent_id=RESULTS_FOLDER_ID,
):
    query = (
        f"'{escape_drive_query(parent_id)}' in parents "
        f"and name = '{escape_drive_query(name)}' "
        "and trashed = false"
    )

    response = (
        drive_service.files()
        .list(
            q=query,
            spaces="drive",
            fields=(
                "files("
                "id,name,mimeType,"
                "createdTime,modifiedTime,size"
                ")"
            ),
            orderBy="modifiedTime desc",
            pageSize=10,
        )
        .execute()
    )

    files = response.get("files", [])
    return files[0] if files else None


def download_drive_bytes(file_id):
    request = (
        drive_service.files()
        .get_media(fileId=file_id)
    )
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(
        buffer,
        request,
    )

    done = False
    while not done:
        _, done = downloader.next_chunk()

    return buffer.getvalue()


def download_drive_json(file_id):
    return json.loads(
        download_drive_bytes(
            file_id
        ).decode("utf-8")
    )


def upload_bytes(
    name,
    content,
    mime_type,
    parent_id=RESULTS_FOLDER_ID,
    replace=True,
):
    existing = find_drive_file(
        name,
        parent_id,
    )

    media = MediaIoBaseUpload(
        io.BytesIO(content),
        mimetype=mime_type,
        resumable=False,
    )

    if existing and replace:
        return (
            drive_service.files()
            .update(
                fileId=existing["id"],
                media_body=media,
                fields="id,name,modifiedTime",
            )
            .execute()
        )

    metadata = {
        "name": name,
        "parents": [parent_id],
    }

    return (
        drive_service.files()
        .create(
            body=metadata,
            media_body=media,
            fields="id,name,modifiedTime",
        )
        .execute()
    )


def upload_text(
    name,
    text,
    mime_type="text/plain",
    parent_id=RESULTS_FOLDER_ID,
    replace=True,
):
    return upload_bytes(
        name,
        text.encode("utf-8"),
        mime_type,
        parent_id,
        replace,
    )


def delete_drive_file(file_id):
    (
        drive_service.files()
        .delete(fileId=file_id)
        .execute()
    )


def read_existing_result():
    result_file = find_drive_file(
        FINAL_RESULT_NAME
    )
    done_lock = find_drive_file(
        DONE_LOCK_NAME
    )

    if result_file:
        result = download_drive_json(
            result_file["id"]
        )

        saved_hash = result.get(
            "config_hash"
        )

        if (
            saved_hash
            and saved_hash != CONFIG_HASH
        ):
            raise RuntimeError(
                "Drive의 기존 final_test_result.json이 "
                "현재 Stage 5 LOCKED_CONFIG와 다릅니다."
            )

        print(
            "이미 완료된 Stage 5 official test "
            "결과를 불러왔습니다."
        )

        display(
            pd.DataFrame(
                [
                    {
                        **result[
                            "test_metrics"
                        ],
                        "misc_f1": result[
                            "talk.religion.misc"
                        ]["f1"],
                    }
                ]
            ).style.format(
                precision=4
            )
        )

        return result

    if done_lock:
        raise RuntimeError(
            "완료 lock은 있지만 "
            "final_test_result.json이 없습니다."
        )

    return None


def acquire_test_lock():
    running = find_drive_file(
        RUNNING_LOCK_NAME
    )

    if running:
        if FORCE_REMOVE_RUNNING_LOCK:
            delete_drive_file(
                running["id"]
            )
        else:
            raise RuntimeError(
                "Stage 5 running lock이 있습니다. "
                "다른 실행이 끝났는지 확인한 뒤, "
                "비정상 종료가 확실할 때만 "
                "FORCE_REMOVE_RUNNING_LOCK=True로 "
                "재실행하세요."
            )

    payload = {
        "status": "running",
        "config_hash": CONFIG_HASH,
        "started_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
    }

    created = upload_text(
        RUNNING_LOCK_NAME,
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
        ),
        mime_type="application/json",
        replace=False,
    )

    print("Stage 5 test lock:", created["id"])


def release_running_lock():
    running = find_drive_file(
        RUNNING_LOCK_NAME
    )

    if running:
        delete_drive_file(
            running["id"]
        )


# Stage 5와 동일한 데이터 및 전처리


In [4]:
TOKEN_PATTERN = re.compile(
    r"[a-z]+(?:'[a-z]+)?"
)

UUENCODE_BEGIN_PATTERN = re.compile(
    r"^\s*begin\s+[0-7]{3}\s+\S+",
    flags=re.IGNORECASE,
)
UUENCODE_END_PATTERN = re.compile(
    r"^\s*end\s*$",
    flags=re.IGNORECASE,
)
BASE64_LINE_PATTERN = re.compile(
    r"^[A-Za-z0-9+/=]+$"
)


def tokenize_original(text):
    text = str(text).lower()

    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text,
    )
    text = re.sub(
        r"\b[\w.\-+]+@[\w.\-]+\.\w+\b",
        " ",
        text,
    )
    text = re.sub(
        r"[^a-z'\s]",
        " ",
        text,
    )
    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return TOKEN_PATTERN.findall(text)


def looks_like_encoded_line(line):
    stripped = line.strip()

    if len(stripped) < 80:
        return False

    compact = re.sub(
        r"\s+",
        "",
        stripped,
    )

    if len(compact) < 80:
        return False

    base64_like = (
        len(compact) >= 100
        and BASE64_LINE_PATTERN.fullmatch(
            compact
        )
        is not None
    )

    symbol_count = sum(
        not character.isalnum()
        for character in compact
    )
    symbol_ratio = (
        symbol_count
        / max(len(compact), 1)
    )

    uuencode_like = (
        stripped.startswith("M")
        and len(compact) >= 60
        and symbol_ratio >= 0.20
    )

    symbol_heavy = (
        len(compact) >= 120
        and symbol_ratio >= 0.35
    )

    return (
        base64_like
        or uuencode_like
        or symbol_heavy
    )


def remove_encoded_payload(text):
    retained_lines = []
    inside_uuencode_block = False

    for line in str(text).splitlines():
        if UUENCODE_BEGIN_PATTERN.match(
            line
        ):
            inside_uuencode_block = True
            continue

        if inside_uuencode_block:
            if UUENCODE_END_PATTERN.match(
                line
            ):
                inside_uuencode_block = False
            continue

        if looks_like_encoded_line(line):
            continue

        retained_lines.append(line)

    return "\n".join(retained_lines)


def tokenize_clean(text):
    text = remove_encoded_payload(text)
    tokens = tokenize_original(text)
    token_counts = Counter(tokens)

    repeated_short_noise = {
        token
        for token, count
        in token_counts.items()
        if (
            len(token) <= 2
            and count >= 20
            and (
                count
                / max(len(tokens), 1)
            ) >= 0.20
        )
    }

    if token_counts.get("ax", 0) >= 8:
        repeated_short_noise.add("ax")

    if repeated_short_noise:
        tokens = [
            token
            for token in tokens
            if token
            not in repeated_short_noise
        ]

    return tokens


def make_clean_corpus(
    text_array,
    desc,
):
    return np.asarray(
        [
            " ".join(
                tokenize_clean(text)
            )
            for text in tqdm(
                text_array,
                desc=desc,
            )
        ],
        dtype=object,
    )


def load_stage5_fixed_split():
    news_data = fetch_20newsgroups(
        subset="all",
        remove=(
            "headers",
            "footers",
            "quotes",
        ),
        shuffle=True,
        random_state=SEED,
    )

    texts = np.asarray(
        news_data.data,
        dtype=object,
    )
    labels = np.asarray(
        news_data.target,
        dtype=np.int64,
    )
    target_names = list(
        news_data.target_names
    )

    (
        cv_texts,
        test_texts,
        cv_labels,
        test_labels,
    ) = train_test_split(
        texts,
        labels,
        test_size=TEST_SIZE,
        stratify=labels,
        random_state=SEED,
    )

    test_hash = hashlib.sha256(
        np.asarray(
            test_labels,
            dtype=np.int64,
        ).tobytes()
        + "\n".join(
            str(text)
            for text in test_texts
        ).encode("utf-8")
    ).hexdigest()

    return {
        "cv_texts": cv_texts,
        "test_texts": test_texts,
        "cv_labels": cv_labels,
        "test_labels": test_labels,
        "target_names": target_names,
        "test_hash": test_hash,
    }


In [5]:
def build_feature_union():
    return FeatureUnion(
        [
            (
                "word",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_df=0.98,
                    sublinear_tf=True,
                    max_features=(
                        WORD_MAX_FEATURES
                    ),
                    dtype=np.float32,
                ),
            ),
            (
                "char",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    min_df=2,
                    sublinear_tf=True,
                    max_features=(
                        CHAR_MAX_FEATURES
                    ),
                    dtype=np.float32,
                ),
            ),
        ],
        n_jobs=min(
            2,
            os.cpu_count() or 1,
        ),
    )


def stable_softmax(scores):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )
    scores -= scores.max(
        axis=1,
        keepdims=True,
    )
    exp_scores = np.exp(scores)

    return (
        exp_scores
        / exp_scores.sum(
            axis=1,
            keepdims=True,
        )
    ).astype(np.float32)


def compute_metrics(
    y_true,
    y_pred,
):
    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        )
    )

    return {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_precision": float(
            precision
        ),
        "macro_recall": float(
            recall
        ),
        "macro_f1": float(f1),
    }


def class_metrics(
    y_true,
    y_pred,
    class_id,
    class_name,
):
    true_binary = (
        y_true == class_id
    ).astype(np.int64)
    pred_binary = (
        y_pred == class_id
    ).astype(np.int64)

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            true_binary,
            pred_binary,
            average="binary",
            zero_division=0,
        )
    )

    return {
        "class": class_name,
        "precision": float(
            precision
        ),
        "recall": float(recall),
        "f1": float(f1),
        "support": int(
            true_binary.sum()
        ),
    }


def make_top_confusions(
    y_true,
    y_pred,
    target_names,
):
    matrix = confusion_matrix(
        y_true,
        y_pred,
    )
    rows = []

    for true_id in range(
        len(target_names)
    ):
        for pred_id in range(
            len(target_names)
        ):
            if true_id == pred_id:
                continue

            count = int(
                matrix[
                    true_id,
                    pred_id,
                ]
            )

            if count > 0:
                rows.append(
                    {
                        "true_class": (
                            target_names[
                                true_id
                            ]
                        ),
                        "predicted_class": (
                            target_names[
                                pred_id
                            ]
                        ),
                        "count": count,
                    }
                )

    return (
        pd.DataFrame(rows)
        .sort_values(
            "count",
            ascending=False,
        )
        .reset_index(drop=True)
    )


# Official test 1회 실행


In [6]:
def run_stage5_official_test_once():
    existing = read_existing_result()

    if existing is not None:
        return existing

    acquire_test_lock()

    try:
        data = load_stage5_fixed_split()

        print(
            "CV pool:",
            len(data["cv_labels"]),
        )
        print(
            "Official test:",
            len(data["test_labels"]),
        )

        cv_clean_texts = make_clean_corpus(
            data["cv_texts"],
            "Stage 5 CV preprocessing",
        )
        test_clean_texts = (
            make_clean_corpus(
                data["test_texts"],
                "Stage 5 test preprocessing",
            )
        )

        feature_extractor = (
            build_feature_union()
        )

        print("CV TF-IDF fit...")
        train_features = (
            feature_extractor.fit_transform(
                cv_clean_texts
            )
        )

        print("Test TF-IDF transform...")
        test_features = (
            feature_extractor.transform(
                test_clean_texts
            )
        )

        classifier = LinearSVC(
            C=1.0,
            dual=True,
            max_iter=5_000,
            tol=1e-4,
            random_state=(
                SEED + 99_999
            ),
        )

        print(
            "CV pool 전체 LinearSVC fit..."
        )
        classifier.fit(
            train_features,
            data["cv_labels"],
        )

        print(
            "Official test 1회 평가..."
        )
        scores = (
            classifier.decision_function(
                test_features
            )
        )
        probabilities = stable_softmax(
            scores
        )
        predictions = scores.argmax(
            axis=1
        )

        metrics = compute_metrics(
            data["test_labels"],
            predictions,
        )

        misc_name = (
            "talk.religion.misc"
        )
        misc_id = (
            data["target_names"].index(
                misc_name
            )
        )
        misc = class_metrics(
            data["test_labels"],
            predictions,
            misc_id,
            misc_name,
        )

        report_df = pd.DataFrame(
            classification_report(
                data["test_labels"],
                predictions,
                target_names=(
                    data["target_names"]
                ),
                output_dict=True,
                zero_division=0,
            )
        ).T

        confusions_df = (
            make_top_confusions(
                data["test_labels"],
                predictions,
                data["target_names"],
            )
        )

        result = {
            "status": "complete",
            "stage": 5,
            "completed_at_utc": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
            "config_hash": CONFIG_HASH,
            "locked_config": (
                LOCKED_CONFIG
            ),
            "test_hash": (
                data["test_hash"]
            ),
            "documents": int(
                len(data["test_labels"])
            ),
            "test_metrics": metrics,
            "talk.religion.misc": misc,
        }

        upload_text(
            FINAL_RESULT_NAME,
            json.dumps(
                result,
                ensure_ascii=False,
                indent=2,
            ),
            mime_type=(
                "application/json"
            ),
        )
        upload_text(
            FINAL_REPORT_NAME,
            report_df.to_csv(),
            mime_type="text/csv",
        )
        upload_text(
            FINAL_CONFUSIONS_NAME,
            confusions_df.to_csv(
                index=False
            ),
            mime_type="text/csv",
        )

        probability_buffer = (
            io.BytesIO()
        )
        np.save(
            probability_buffer,
            probabilities,
        )
        upload_bytes(
            FINAL_PROBABILITIES_NAME,
            probability_buffer.getvalue(),
            mime_type=(
                "application/octet-stream"
            ),
        )

        if SAVE_MODEL_TO_DRIVE:
            model_path = Path(
                "/content/"
                "mission10_stage5_final.joblib"
            )
            joblib.dump(
                {
                    "feature_extractor": (
                        feature_extractor
                    ),
                    "classifier": (
                        classifier
                    ),
                    "target_names": (
                        data[
                            "target_names"
                        ]
                    ),
                    "config": (
                        LOCKED_CONFIG
                    ),
                },
                model_path,
                compress=3,
            )
            upload_bytes(
                model_path.name,
                model_path.read_bytes(),
                mime_type=(
                    "application/octet-stream"
                ),
            )

        done_payload = {
            "status": "done",
            "stage": 5,
            "config_hash": CONFIG_HASH,
            "test_hash": (
                data["test_hash"]
            ),
            "completed_at_utc": (
                result[
                    "completed_at_utc"
                ]
            ),
        }

        upload_text(
            DONE_LOCK_NAME,
            json.dumps(
                done_payload,
                ensure_ascii=False,
                indent=2,
            ),
            mime_type=(
                "application/json"
            ),
        )

        release_running_lock()

        print("=" * 72)
        print("STAGE 5 OFFICIAL TEST COMPLETE")
        print(
            json.dumps(
                result,
                ensure_ascii=False,
                indent=2,
            )
        )
        print("=" * 72)

        display(
            pd.DataFrame(
                [
                    {
                        **metrics,
                        "misc_precision": (
                            misc["precision"]
                        ),
                        "misc_recall": (
                            misc["recall"]
                        ),
                        "misc_f1": (
                            misc["f1"]
                        ),
                    }
                ]
            ).style.format(
                precision=4
            )
        )

        return result

    except Exception:
        release_running_lock()
        raise


FINAL_TEST_RESULT = (
    run_stage5_official_test_once()
)


Stage 5 test lock: 1OaOvMD99PFqUtvWKojWUuPyJIwd79GCx
CV pool: 16019
Official test: 2827


Stage 5 CV preprocessing:   0%|          | 0/16019 [00:00<?, ?it/s]

Stage 5 test preprocessing:   0%|          | 0/2827 [00:00<?, ?it/s]

CV TF-IDF fit...
Test TF-IDF transform...
CV pool 전체 LinearSVC fit...
Official test 1회 평가...
STAGE 5 OFFICIAL TEST COMPLETE
{
  "status": "complete",
  "stage": 5,
  "completed_at_utc": "2026-07-27T04:41:03.961870+00:00",
  "config_hash": "c33995409eb8b135ce07a25c841c75513c3968742bc0a3e4c5b435dcfb72446f",
  "locked_config": {
    "stage": 5,
    "input": "body_only_header_footer_quotes_removed",
    "preprocessing": "encoded_payload_and_repeated_short_noise_removal",
    "classifier": "linear_svc",
    "use_specialist": false,
    "C": 1.0,
    "seed": 42,
    "test_size": 0.15,
    "word_ngram_range": [
      1,
      2
    ],
    "char_ngram_range": [
      3,
      5
    ],
    "word_max_features": 80000,
    "char_max_features": 120000
  },
  "test_hash": "e1941059324345d7edfc6c748ac1a805da0229d568f6f7e4796cbc06529f92a4",
  "documents": 2827,
  "test_metrics": {
    "accuracy": 0.7707817474354439,
    "macro_precision": 0.7785794683511317,
    "macro_recall": 0.7622790695780624,
  

,accuracy,macro_precision,macro_recall,macro_f1,misc_precision,misc_recall,misc_f1
0,0.7708,0.7786,0.7623,0.7659,0.7143,0.4787,0.5732


## 재실행 규칙

- 정상 완료 후 다시 실행하면 Drive의 기존 결과만 표시합니다.
- 세션이 강제 종료되어 `stage5_official_test_running.lock`만 남은 경우, 다른 실행이 없는지 확인한 뒤 `FORCE_REMOVE_RUNNING_LOCK=True`로 한 번 재실행합니다.
- Stage 6 test notebook과 결과 폴더가 다르므로 두 test는 서로 충돌하지 않습니다.
